# Fraud Risk — LoRA Fine-tuning (Colab)

Fine-tunes `google/gemma-2b` with LoRA on the 200-row balanced fundraiser dataset  
for binary fraud classification. Replaces the GCP Compute Engine GPU VM workflow.

**Runtime:** Connect to a T4 GPU runtime (`Runtime → Change runtime type → T4 GPU`).

**HuggingFace prerequisite:** Accept the Gemma license at https://huggingface.co/google/gemma-2b,  
then set your token in the cell below.

**Output:** LoRA adapter saved to `gs://fraud-risk-models/lora-adapters/v1/final-adapter`

In [6]:
# ── 1. Install deps & authenticate ────────────────────────────────────────────
!pip install -q peft transformers accelerate datasets mlflow google-cloud-storage

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# GCP auth (opens a browser prompt — click the link, paste the code)
from google.colab import auth
auth.authenticate_user()

# HuggingFace token — required for gated Gemma model
# Get yours at https://huggingface.co/settings/tokens
from huggingface_hub import login
from google.colab import userdata
HF_TOKEN = userdata.get('HF_token')
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
else:
    raise ValueError("Set HF_TOKEN above — Gemma requires a HuggingFace token.")

print("Setup complete.")

SecretNotFoundError: Secret HF_TOKEN does not exist.

In [ ]:
# ── 2. Config (mirrors ml_training_service/configs/lora_config.yaml) ──────────
GCP_PROJECT       = ""   # your GCP project ID, e.g. "my-fraud-project"
GCS_DATA_PATH     = "gs://fraud-risk-data/technique2_train_balanced_200.csv"
GCS_ADAPTER_OUT   = "gs://fraud-risk-models/lora-adapters/v1"
OUTPUT_DIR        = "/content/lora-adapter"

BASE_MODEL        = "google/gemma-2b"
LORA_R            = 16
LORA_ALPHA        = 32
LORA_DROPOUT      = 0.1
LEARNING_RATE     = 2e-4
NUM_EPOCHS        = 10
BATCH_SIZE        = 8
MAX_LENGTH        = 512

In [ ]:
# ── 3. Load training data ─────────────────────────────────────────────────────
import pandas as pd

def load_data():
    """Try GCS first; fall back to manual Colab file upload."""
    try:
        df = pd.read_csv(GCS_DATA_PATH, storage_options={"token": "google_default"})
        print(f"Loaded {len(df)} rows from GCS: {GCS_DATA_PATH}")
        return df
    except Exception as e:
        print(f"GCS unavailable ({e})")
        print("Upload technique2_train_balanced_200.csv from your local machine:")
        from google.colab import files
        uploaded = files.upload()
        df = pd.read_csv(next(iter(uploaded)))
        print(f"Loaded {len(df)} rows from upload")
        return df

df = load_data()
print(f"Columns: {list(df.columns)}")
print(f"Label distribution: {df['label'].value_counts().to_dict()}")

In [ ]:
# ── 4. Prepare records and train/eval split ───────────────────────────────────
from sklearn.model_selection import train_test_split

def build_records(df: pd.DataFrame) -> list[dict]:
    records = []
    for _, row in df.iterrows():
        title = str(row.get("title", "") or "")
        # use description_clean (HTML-stripped); fall back to raw description
        desc = str(row.get("description_clean", "") or row.get("description", "") or "")
        records.append({
            "fund_id":    str(row.get("fund_id", "")),
            "input_text": f"Title: {title}\n\nDescription: {desc}",
            "label":      int(row["label"]),
        })
    return records

all_records = build_records(df)
labels = [r["label"] for r in all_records]

train_records, eval_records = train_test_split(
    all_records, test_size=0.2, random_state=42, stratify=labels
)
print(f"Train: {len(train_records)}  Eval: {len(eval_records)}")
print(f"Sample input (truncated): {train_records[0]['input_text'][:200]}")

In [ ]:
# ── 5. LoRA fine-tuning ───────────────────────────────────────────────────────
import torch
import mlflow
from pathlib import Path
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import roc_auc_score, f1_score

assert torch.cuda.is_available(), "No GPU detected — connect to a T4 runtime first."
print(f"GPU: {torch.cuda.get_device_name(0)}")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    preds = (probs >= 0.5).astype(int)
    return {
        "auc_roc": roc_auc_score(labels, probs),
        "f1":      f1_score(labels, preds, zero_division=0),
    }


def tokenize_dataset(records, tokenizer, max_length):
    ds = Dataset.from_list(records)
    def _tokenize(batch):
        return tokenizer(
            batch["input_text"],
            truncation=True, padding="max_length", max_length=max_length,
        )
    return ds.map(_tokenize, batched=True, remove_columns=["input_text", "fund_id"])


# Build model
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=2,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.config.pad_token_id = tokenizer.pad_token_id

lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

# Tokenize
train_ds = tokenize_dataset(train_records, tokenizer, MAX_LENGTH)
eval_ds  = tokenize_dataset(eval_records,  tokenizer, MAX_LENGTH)

# Train with local MLflow tracking (artifacts pushed to GCS in the next cell)
mlflow.set_tracking_uri(f"file:{OUTPUT_DIR}/mlruns")
mlflow.set_experiment("fraud-lora-colab")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="auc_roc",
    greater_is_better=True,
    fp16=True,
    logging_steps=10,
    report_to="none",
)

with mlflow.start_run():
    mlflow.log_params({
        "base_model": BASE_MODEL, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT, "learning_rate": LEARNING_RATE,
        "num_epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE,
        "train_size": len(train_records), "eval_size": len(eval_records),
    })

    trainer = Trainer(
        model=model, args=training_args,
        train_dataset=train_ds, eval_dataset=eval_ds,
        compute_metrics=compute_metrics,
    )
    trainer.train()

    eval_results = trainer.evaluate()
    print("\nFinal eval:", eval_results)
    mlflow.log_metrics({k.replace("eval_", ""): v for k, v in eval_results.items()})

    adapter_path = f"{OUTPUT_DIR}/final-adapter"
    model.save_pretrained(adapter_path)
    tokenizer.save_pretrained(adapter_path)
    mlflow.log_artifact(adapter_path, artifact_path="lora-adapter")

print(f"\nAdapter saved to: {adapter_path}")

In [ ]:
# ── 6. Push adapter to GCS ────────────────────────────────────────────────────
import subprocess

print(f"Uploading adapter to {GCS_ADAPTER_OUT} ...")
result = subprocess.run(
    ["gsutil", "-m", "cp", "-r", f"{OUTPUT_DIR}/final-adapter", GCS_ADAPTER_OUT],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print("ERROR:", result.stderr)
else:
    print("Done!")
    print(f"\nTo download locally and run evaluation:")
    print(f"  gsutil -m cp -r {GCS_ADAPTER_OUT}/final-adapter models/lora-adapter/")
    print(f"  LORA_ADAPTER_PATH=models/lora-adapter/final-adapter make serve")